<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/DemoRAG%2CEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q chromadb openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
# 1. What vectop database should we use? -> ChromaDB
# 2. What embedding model should we use? -> OpenAI embedding model

# Embedding vector, generated by AI model A cannot be directly compared to another embedding vector, generated by AI model B.

In [3]:
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from google.colab import userdata
from openai import OpenAI
from pydantic import BaseModel

openai_api_key = userdata.get('OPENAI_API_KEY')
openai_client = OpenAI(api_key=openai_api_key)

chromadb_client = PersistentClient("/content/chromadb")
chromadb_collection = chromadb_client.get_or_create_collection(
      name = "books-v3",
      embedding_function = OpenAIEmbeddingFunction(api_key = openai_api_key, model_name = "text-embedding-3-large")
)

In [4]:
import json
from pathlib import Path

def index_file(path_to_file:Path, chunk_size: int = 5, chunk_overlap_size: int = 2, batch_size: int = 100 ) -> None:
  with path_to_file.open() as file_content:
      file_lines = file_content.readlines()
      file_lines = [fl.strip() for fl in file_lines]
      file_lines = [fl for fl in file_lines if fl != '']

      chunks = []
      line_index = 0
      while line_index < len(file_lines):
          start = line_index
          end = line_index + chunk_size

          current = '\n'.join(file_lines[start:end])
          chunks.append({"text": current, "book": path_to_file.stem, "rel_line_start": start + 1, "rel_line_end":end})

          line_index = end - chunk_overlap_size

      chunk_index = 0
      batch_size = 100
      while chunk_index < len(chunks):
          start = chunk_index
          end = chunk_index + batch_size
          current_batch = chunks[start:end]

          # TODO: Id must be "book_name + line_start + line_end"
          # TODO: Add metadata
          # zapisvane v vektornata baza
          chromadb_collection.add(
              ids = [f"{x['book']}_{x['rel_line_start']}_{x['rel_line_end']}"for x in current_batch],
              documents = [x['text'] for x in current_batch],
              metadatas = [{"book": x['book'], "rel_line_start": x['rel_line_start'], 'rel_line_end': x['rel_line_end']} for x in current_batch]
          )

          #print(f"Successfully embedded documents in range [{start}, {end}]")
          chunk_index = end

In [5]:
index_file(Path("/content/Adventures of Sherlock Holmes.txt"))

In [6]:
index_file(Path("/content/Alice's Adventures in Wonderland.txt"))

In [7]:
#[start: end + 1 ]
#file_lines[1:5]

In [8]:
results1 = chromadb_collection.query(
    query_texts = "color green",
    n_results = 5
)

In [9]:
results1

{'ids': [["Alice's Adventures in Wonderland_904_908",
   'Adventures of Sherlock Holmes_5779_5783',
   "Alice's Adventures in Wonderland_1450_1454",
   "Alice's Adventures in Wonderland_901_905",
   'Adventures of Sherlock Holmes_5776_5780']],
 'embeddings': None,
 'documents': [['“What _can_ all that green stuff be?” said Alice. “And where _have_ my\nshoulders got to? And oh, my poor hands, how is it I can’t see you?”\nShe was moving them about as she spoke, but no result seemed to follow,\nexcept a little shaking among the distant green leaves.\nAs there seemed to be no chance of getting her hands up to her head,',
   'way-side hedges were just throwing out their first green shoots, and\nthe air was full of the pleasant smell of the moist earth. To me at\nleast there was a strange contrast between the sweet promise of the\nspring and this sinister quest upon which we were engaged. My companion\nsat in the front of the trap, his arms folded, his hat pulled down over',
   'growing on i

In [10]:
# (args) -> Quote
class QuoteLookupArgs(BaseModel):
  query: str
  # top_n: int

In [11]:
QuoteLookupArgs.model_json_schema()

{'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'QuoteLookupArgs',
 'type': 'object'}

In [12]:
TOOLS = [
    {
        "type": "function",
        "name":"quote_lookup",
        "description": "Call this tool to lookup a quote from the internal collection of books that corresponds semantically to a given natural language query.",
        "parameters": {
            **QuoteLookupArgs.model_json_schema(),
            "additionalProperties": False
        },
        "strict": True,
    }
]

In [13]:
TOOLS

[{'type': 'function',
  'name': 'quote_lookup',
  'description': 'Call this tool to lookup a quote from the internal collection of books that corresponds semantically to a given natural language query.',
  'parameters': {'properties': {'query': {'title': 'Query', 'type': 'string'}},
   'required': ['query'],
   'title': 'QuoteLookupArgs',
   'type': 'object',
   'additionalProperties': False},
  'strict': True}]

In [14]:
def quote_lookup_tool_handler(raw_args: str) -> str:
    typed_args = QuoteLookupArgs.model_validate(json.loads(raw_args))

    search_result = chromadb_collection.query(query_texts=[typed_args.query])
    count = len(search_result["ids"][0])

    output = []

    for i in range(count):
        output.append(f"Match #{i + 1}")

        book = search_result["metadatas"][0][i]["book"]
        output.append(f"Book: {book}")

        quote = search_result["documents"][0][i]
        output.append(f"Quote: {quote}")

        output.append("")

    return "\n".join(output)


TOOL_HANDLERS = {
    "quote_lookup":quote_lookup_tool_handler
}

In [15]:
print(quote_lookup_tool_handler('{"query":"color theory"}'))

Match #1
Book: Adventures of Sherlock Holmes
Quote: so many in the whole country as were brought together by that single
advertisement. Every shade of color they were—straw, lemon, orange,
brick, Irish-setter, liver, clay; but, as Spaulding said, there were
not many who had the real vivid flame-colored tint. When I saw how many
were waiting, I would have given it up in despair; but Spaulding would

Match #2
Book: Adventures of Sherlock Holmes
Quote: is fond of a particular shade of electric blue, and would like
you to wear such a dress in-doors in the morning. You need not,
however, go to the expense of purchasing one, as we have one
belonging to my dear daughter Alice (now in Philadelphia), which
would, I should think, fit you very well. Then, as to sitting

Match #3
Book: Alice's Adventures in Wonderland
Quote: growing on it were white, but there were three gardeners at it, busily
painting them red. Alice thought this a very curious thing, and she
went nearer to watch them, and just 

In [31]:
def extract_tool_calls(response):
  return [x for x in response.output if x.type == 'function_call']

def agent_loop(system_prompt: str, user_prompt: str, model: str = "gpt-5.4-mini") -> str:
    context = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content":user_prompt}
    ]

    response_create_kwargs= {"model": model, "tools": TOOLS, "reasoning": {"effort":"high"}}
    response = openai_client.responses.create(input = context, **response_create_kwargs)
    tool_calls = extract_tool_calls(response)

    while tool_calls:
        print(f"Discovered {len(tool_calls)} tool calls should be executed.")
        for tc in tool_calls:
            print(tc)
        print()

        context.extend(response.output)

        for tc in tool_calls:
            handler = TOOL_HANDLERS[tc.name]
            current_result = handler(tc.arguments)

            context.append({"type":"function_call_output", "call_id":tc.call_id, "output": current_result})

        response = openai_client.responses.create(input = context, **response_create_kwargs)
        tool_calls = extract_tool_calls(response)

    return response.output_text

In [33]:
print(
    agent_loop(
        system_prompt = "You are a helpful assistant for book analysis and quote finding. The user will enter a given topic and you need to use the \"quote_lookup\" tool to find relevant quotes from the internal collection of books. Use it multiple times with different (similar) variations of the original user query.",
        user_prompt = "Find quotes relevant to color theory, then if bright colors are ore likely, use the \"query_lookup\" tool again to search for love and affection; if not use the \"query_lookup\" tool to search for drama."
    )
)

Discovered 4 tool calls should be executed.
ResponseFunctionToolCall(arguments='{"query":"color theory"}', call_id='call_Cb7JpKHy4HpKtExq9rNKvLkT', name='quote_lookup', type='function_call', id='fc_020fe3ef29c39805006a5a9424e228819bb9a251fb8ce845ac', caller=None, namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"theory of color and hue"}', call_id='call_zsxMIFHkTcf5mGRi71VSVpVv', name='quote_lookup', type='function_call', id='fc_020fe3ef29c39805006a5a9424e23c819bbbc109a6d547dee2', caller=None, namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"color harmony, palette, and pigments"}', call_id='call_M27DJeku36GImfLwU1tzS6cL', name='quote_lookup', type='function_call', id='fc_020fe3ef29c39805006a5a9424e244819ba87d2798c8bbb0b1', caller=None, namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"bright colors and color mixing"}', call_id='call_tq3fGpKfbclHcZDRUxHk6f5m', name='quote_lookup', type='func

In [26]:
print(
    agent_loop(
        system_prompt = "You are a helpful assistant for book analysis and quote finding. The user will enter a given topic and you need to use the \"quote_lookup\" tool to find relevant quotes from the internal collection of books. Use it multiple times with different (similar) variations of the original user query.",
        user_prompt = "Search extensively for quotes relevant to deep, special, psychological moments, followed by happines or sadness. Than repeat again in the other way around."
    )
)

Discovered 6 tool calls should be executed.
ResponseFunctionToolCall(arguments='{"query":"deep special psychological moment followed by happiness quote"}', call_id='call_hbd1b5wjRRIyb5QNPOMoUmQx', name='quote_lookup', type='function_call', id='fc_03c5ab3454a32650006a5a9274489c819b9508f54e70554b96', caller=None, namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"deep special psychological moment followed by sadness quote"}', call_id='call_cJdyK7eENj2YDPuIfExR4tWp', name='quote_lookup', type='function_call', id='fc_03c5ab3454a32650006a5a927448b0819bb0a571b2e42a4aff', caller=None, namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"happiness after a deep psychological moment quote"}', call_id='call_xazuVEdciJLPLI0aKHjEl2Xb', name='quote_lookup', type='function_call', id='fc_03c5ab3454a32650006a5a927448c0819bb5dcd084c11d69ea', caller=None, namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"sadness a